# Day 31 — 교란변수: 그 차이는 정말 "군" 때문일까

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-31.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-31.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 두 군은 진단 말고도 다른가: 나이와 성별 점검

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

# 참가자 단위 표 (나이·성별은 사람마다 하나이므로 groupby 키에 넣는다)
person = df.groupby([ID_COL, GROUP_COL, "sex"], as_index=False)[["age"] + FEATURES].mean()

print("군별 나이 (참가자 단위):")
print(person.groupby(GROUP_COL)["age"].describe()[["count", "mean", "std", "min", "max"]].round(1))
print()
print("군별 성별 인원:")
print(pd.crosstab(person[GROUP_COL], person["sex"]))
print()
print("나이와 각 feature의 상관계수 (참가자 단위):")
print(person[FEATURES].corrwith(person["age"]).round(2))

### 예시 2 — 회귀로 나이를 보정하면 군 차이가 어떻게 변하는가

예시 1에 이어서 실행합니다. `statsmodels`의 수식 문법(Day 11의 설계식과 같은 모양)을 씁니다.

In [ ]:
import statsmodels.formula.api as smf

# 대조군(HC)을 기준으로 삼는다: 계수는 "PD - HC"
group_term = f"C({GROUP_COL}, Treatment('HC'))"
key = f"{group_term}[T.PD]"

for feature in ["stride_time_cv", "tremor_band_power"]:
    crude = smf.ols(f"{feature} ~ {group_term}", data=person).fit()
    adjusted = smf.ols(f"{feature} ~ {group_term} + age", data=person).fit()
    lo0, hi0 = crude.conf_int().loc[key]
    lo1, hi1 = adjusted.conf_int().loc[key]
    print(feature)
    print(f"  보정 전 군 차이  = {crude.params[key]:.2f}  95% CI [{lo0:.2f}, {hi0:.2f}]")
    print(f"  나이 보정 후     = {adjusted.params[key]:.2f}  95% CI [{lo1:.2f}, {hi1:.2f}]")
    print(f"  나이 1살당 변화  = {adjusted.params['age']:.3f}")

### 예시 3 — 반복 측정을 독립 표본처럼 쓰면 생기는 일

예시 2에 이어서 실행합니다. 같은 feature를 "행 300개"로 검정할 때와 "참가자 50명"으로 검정할 때를 나란히 놓습니다.

In [ ]:
from scipy import stats

print(f"{'feature':<20}{'p_rows (n=300)':>16}{'p_person (n=50)':>18}")   # 행 단위 / 참가자 단위
for f in ["tremor_band_power", "step_regularity", "jerk_mean"]:
    p_rows = stats.ttest_ind(df.loc[df[GROUP_COL] == "PD", f],
                             df.loc[df[GROUP_COL] == "HC", f], equal_var=False).pvalue
    p_person = stats.ttest_ind(person.loc[person[GROUP_COL] == "PD", f],
                               person.loc[person[GROUP_COL] == "HC", f], equal_var=False).pvalue
    print(f"{f:<20}{p_rows:>16.2g}{p_person:>18.3f}")